In [ ]:
import os
import re
import pypdf
import ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Config
PDF_PATH = "de.pdf" 
MODEL = "llama3.2" 
OUTPUT_MD = "de_interview_prep.md"

chapters_list = [
    """Chapter 1. Data Engineering""",
    """Chapter 2. The Data
Engineering Lifecycle""",
    """Chapter 3. Designing Good
Data Architecture""",
    """Chapter 4. Choosing
Technologies Across the Data
Engineering Lifecycle""",
"""Chapter 5. Data Generation in
Source Systems""",
"""Chapter 6. Storage""",
"""Chapter 7. Ingestion""",
"""Chapter 8. Queries, Modeling, and
Transformation""",
"""Chapter 9. Serving Data for
Analytics, Machine Learning,
and Reverse ETL""",
"""Chapter 10. Security and
Privacy""",
"""Chapter 11. The Future of Data
Engineering"""

]
_chapters_list = [
    'Chapter 1.',
    'Chapter 2.',
    'Chapter 3.',
    'Chapter 4.',
    'Chapter 5.',
    'Chapter 6.',
    'Chapter 7.',
    'Chapter 8.',
    'Chapter 9.',
    'Chapter 10.',
    'Chapter 11.'

]
def get_chapter_ranges(pdf_path, chapters):
    """Finds the start and end page for each chapter title."""
    reader = pypdf.PdfReader(pdf_path)
    chapter_pages = []
    
    print("Mapping chapters to page numbers...")
    for title in chapters:
        found_page = -1
        # Search for the title string in the PDF text
        for i, page in enumerate(reader.pages):
            if title.lower() in page.extract_text().lower():
                found_page = i 
                break
        chapter_pages.append({"title": title, "start": found_page})

    # Calculate end pages based on the next chapter's start
    for i in range(len(chapter_pages)):
        if chapter_pages[i]["start"] == -1:
            continue
        if i + 1 < len(chapter_pages) and chapter_pages[i+1]["start"] != -1:
            chapter_pages[i]["end"] = chapter_pages[i+1]["start"]
        else:
            chapter_pages[i]["end"] = len(reader.pages)
            
    return [c for c in chapter_pages if c["start"] != -1]

def extract_text_from_range(pdf_path, start_page, end_page):
    """Extracts text only within the specified page boundaries."""
    text = ""
    try:
        reader = pypdf.PdfReader(pdf_path)
        for i in range(start_page, end_page):
            extracted = reader.pages[i].extract_text()
            if extracted:
                text += extracted + "\n"
    except Exception as e:
        print(f"Error reading range {start_page}-{end_page}: {e}")
    return text

def extract_text(pdf_path):
    """Fixed: Uses pypdf correctly and handles empty pages."""
    text = ""
    try:
        reader = pypdf.PdfReader(pdf_path)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    except Exception as e:
        print(f"Error reading PDF: {e}")
    return text

def detect_chapters(text):
    """Fixed: Correctly pairs detected titles with their following content."""
    # Using a capturing group keeps the titles in the split list
    parts = re.split(r'(Chapter \d+|CHAPTER \d+|Section \d+)', text)
    
    chapters = []
    # Skip index 0 (intro text) and step by 2 to get [Title, Content] pairs
    for i in range(1, len(parts), 2):
        title = parts[i]
        content = parts[i+1] if i+1 < len(parts) else ""
        chapters.append(f"{title}\n{content}")
    
    return chapters[:20]

def summarize_chapter(chapter_text, chapter_num, chapter_title):
    """Maintains your exact prompt while looping through text chunks."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=500)
    chunks = splitter.split_text(chapter_text)
    
    
    prompt_template = f"""
    You are a Staff/Principal Data Engineering architect and interviewer. Write like someone who has built and operated production data platforms.

    Your task is to produce a deeply explanatory, interview‑grade chapter summary. This is NOT a textbook summary. Every concept must be justified through architectural reasoning, trade-offs, and real production consequences.

    Hard requirements:
    - For EVERY concept/pattern/tool/best practice, include at least ONE enterprise example.
    - Examples MUST include: industry + scale (volume/throughput/users) + constraint (SLA, cost, compliance) + failure mode avoided + measurable outcome.
    - Use concrete nouns: data sources (ERP/CRM/web events/IoT), workloads (ETL/ELT/streaming/ML), org context (multi-team, self-serve, governance).
    - If an example would be speculative, label it “Representative enterprise example”.

    Structure your output EXACTLY as follows:

    ## Chapter {chapter_num} {chapter_title} Summary

    ### Core Concepts and Their Rationale
    For EACH major concept introduced, include:
    - **What it is** (1–2 lines, no textbook definitions)
    - **Why it exists** (real engineering pain it solved)
    - **What breaks without it** (data quality, reliability, cost, governance)
    - **Enterprise example** (industry + scale + constraint + outcome)
    - **Anti-example** (when it becomes overkill or harmful)

    ### Architectures and Design Choices
    For EACH architecture or pattern:
    - **Why it is chosen** over simpler designs
    - **Trade-offs** (explicit: cost vs complexity, latency vs correctness)
    - **Enterprise example** (with scale + SLA + compliance/governance angle)
    - **Failure modes** (what goes wrong in real orgs)
    - **When NOT to use it** (team maturity, budget, use case mismatch)
    - **Interview defense** (how you justify it to an interviewer)

    ### Tools and Platforms (ADF, Databricks, Fabric, Spark, etc.)
    For EACH tool:
    - **Why it exists** (operational pain it abstracts)
    - **Enterprise example** (migration, modernization, multi-team platform)
    - **Why it beats alternatives** (and where it loses)
    - **Cost / governance risks** (runaway compute, poor lineage, secrets, RBAC)
    - **Common misuse** that causes outages or bad data

    ### Best Practices Explained (With Consequences)
    For EACH best practice:
    - **Why it’s non-negotiable at scale**
    - **Enterprise example** (what incident it prevented or fixed)
    - **Failure mode if ignored** (silent corruption, SLA miss, audit failure)
    - **How seniors enforce it** (guardrails, templates, policies, CI/CD)

    ### Interview-Level Insights (Decision-Oriented)
    - 6–10 high-signal “WHY” interview questions from this chapter
    - Strong answers that reference:
    - trade-offs
    - constraints
    - examples
    - failure modes
    - Red flags interviewers look for

    Guidelines:
    - Assume the reader already knows definitions
    - Prefer decision logic and consequences over lists
    - Use crisp bullets, but allow short prose if needed
    - 500–900 words if needed to keep examples strong
    - Output ONLY the markdown above
    """

    summary = ""
    # Process up to 10 chunks to stay within context limits
    for i, chunk in enumerate(chunks[:10]):
        # Swapping the placeholder with the actual chunk text
        current_prompt = prompt_template.replace("{chapter_num}", str(chapter_num))
        current_prompt = current_prompt.replace("[Detailed prose here]", f"Focus on chunk {i}: {chunk}")
        
        response = ollama.generate(model=MODEL, prompt=current_prompt)
        summary += response['response'] + "\n"
        print(f"Summarized chunk {i+1}/{min(10, len(chunks))} for Chapter {chapter_num}...")

    return summary

def generate_mocks(summary):
    """Maintains your exact mock interview prompt."""
    # YOUR ORIGINAL PROMPT
    prompt = f"""
    Based on this chapter summary, generate EXACTLY 3 mock Data Engineering interview questions (behavioral/technical). 
    For each: Question, then curated acceptable answer (concise, eloquent, 150-250 words).
    
    Format as:
    ### Mock Question 1
    **Q:** [Question]
    **A:** [Answer]
    
    Output ONLY markdown.
    """
    # Truncate to ensure the prompt doesn't exceed LLM limits
    response = ollama.generate(model=MODEL, prompt=prompt.replace("this chapter summary", summary[:8000]))
    return response['response']

# Main execution
if __name__ == "__main__":
    if not os.path.exists(PDF_PATH):
        print(f"File not found: {PDF_PATH}")
    else:
        # 1. Map the chapters to pages
        ranges = get_chapter_ranges(PDF_PATH, chapters_list)
        md_content = "# Data Engineering Interview Prep\n\n"

        # 2. Process each range
        for i, item in enumerate(ranges, 1):
            print(f"Processing {item['title']} (Pages {item['start']} to {item['end']})...")
            
            # Extract only the relevant text
            chapter_text = extract_text_from_range(PDF_PATH, item['start'], item['end'])
            
            if not chapter_text.strip():
                continue

            # 3. Summarize and Mock
            chap_summary = summarize_chapter(chapter_text, i, item['title'])
            mocks = generate_mocks(chap_summary)
            print(chap_summary)

            md_content += f"{chap_summary}\n\n{mocks}\n\n---\n\n"
            
            # Incremental save
            with open(OUTPUT_MD, 'w', encoding='utf-8') as f:
                f.write(md_content)

        print(f"Done! Created: {OUTPUT_MD}")

Mapping chapters to page numbers...
Processing Chapter 1. Data Engineering (Pages 17 to 59)...
Summarized chunk 1/10 for Chapter 1...
Summarized chunk 2/10 for Chapter 1...
Summarized chunk 3/10 for Chapter 1...
Summarized chunk 4/10 for Chapter 1...
Summarized chunk 5/10 for Chapter 1...
Summarized chunk 6/10 for Chapter 1...
Summarized chunk 7/10 for Chapter 1...
Summarized chunk 8/10 for Chapter 1...
Summarized chunk 9/10 for Chapter 1...
Summarized chunk 10/10 for Chapter 1...
## Chapter 1 Data Engineering Summary

### Core Concepts and Their Rationale

#### **Data Staging**
What it is: A temporary storage area for data, used as a buffer between data sources and processing pipelines.
Why it exists: To prevent data loss due to downstream pipeline failures or configuration changes.
What breaks without it: Unreliable data processing pipelines, lost business value from stale data.
Enterprise example: Confluent Platform (scale: 1000+ users, constraint: SLA-compliant, outcome: Reduced da